# 08 - Final Output (Plan Section 13)

One settled row per bus for the whole month (`ml.bus_matching_final_pairs`),
built from the pair-level layer (`pair_features.py` / `pair_model.py`)
that replaced `belief.py`'s hand-set trust weights. Every bus that ran
in November 2023 gets a row -- an explicit `method` says how it was
settled, following the same never-silently-drop convention Section 9's
`ml.bus_matching_global_assignment` already uses, just at the month
grain instead of the bus-date grain.

**Split detection.** Labeling surfaced a real pattern the one-device-per-bus
assumption cannot represent on its own: a device genuinely gets swapped
for another mid-month. Four such buses were caught by hand while
labeling (12225, 12502, 35252, 35403) and marked `verdict='unsure'`
rather than forced into a single wrong answer. `final_output.detect_split`
tells a real swap apart from a genuine unknown purely from the day-score
time series -- one confident device, a single clean changeover, a
different confident device -- and is verified below to reproduce the
exact date boundaries found by hand on all four.

**Coverage, confirmed live before this run**: of 1,481 in-scope buses
(1,730 that ran, minus 249 structurally excluded), 1,424 were resolved
at >=0.90 confidence with unanimous margin -- every resolved bus's
runner-up scored at least 0.9 below it, so "confident" and "unambiguous"
coincide in practice. The remaining tail is not unexplained: 48 buses
never got a single blocking candidate, and a handful have candidates but
no real evidence for any of them. Both get an explicit bucket below
rather than being left to look like silent failures.


In [1]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))
sys.path.insert(0, str(_root / "ml" / "bus_matching_model" / "app"))

In [2]:
import time

import db
import exclusions
import final_output
import pair_features
import pair_model
import pandas as pd
import psycopg
import schema
import training
from features import DAY_FEATURE_NAMES

from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
schema.ensure_schema(conn)
print("ml schema ready")

ml schema ready


## Rebuild the day model, then the pair model

Retrained fresh from current labels rather than loaded from a stale
run, for the same reason every verification query in this project has
done it this way: the day-score inputs to the pair layer must reflect
every trip label on file *right now*, not whatever existed when some
earlier run was saved.


In [3]:
start = time.monotonic()

root = "ml/bus_matching_model/artifacts"
day = pd.concat(
    [pd.read_parquet(f) for f in sorted(Path(root).glob("features_v2/*.parquet"))],
    ignore_index=True,
)
excluded = exclusions.excluded_bus_ids(conn)
day_in_scope = day[~day["bus_id"].isin(excluded)]

expanded = db.expand_labels_to_training_rows(conn, day_in_scope, None)
day_model = training.train_model(
    expanded[DAY_FEATURE_NAMES], expanded["label"].to_numpy()
)
# Scored on the FULL (unfiltered) frame, not just day_in_scope: split
# detection later needs day_score for every candidate of an unsure
# bus, and it is simplest to have one column that always covers
# whatever `day` is sliced to downstream.
day["day_score"] = training.predict_positive_proba(day_model, day[DAY_FEATURE_NAMES])
day_in_scope = day[~day["bus_id"].isin(excluded)]

n_trip_labels = db.trip_label_counts(conn)["total"]
print(
    f"day model trained on {len(expanded)} rows from {n_trip_labels} trip labels "
    f"in {time.monotonic() - start:.1f}s"
)

day model trained on 1742 rows from 323 trip labels in 5.1s


In [4]:
pairs = pair_features.build_pair_features(conn, day_in_scope, day_in_scope["day_score"])
labels = pair_model.fetch_pair_labels(conn)
print(
    f"{len(pairs)} candidate pairs, {len(labels)} pair labels "
    f"({(labels['verdict'] == 'correct').sum()} correct, "
    f"{(labels['verdict'] == 'unsure').sum()} unsure)"
)

rows = pair_model.build_training_rows(pairs, labels)
pair_state = pair_model.train_pair_model(rows)
if pair_state is None:
    msg = "not enough labeled pairs to train the pair model"
    raise RuntimeError(msg)

m, cv = pair_state["metrics"], pair_state.get("cv", {})
print(
    f"pair model: train={pair_state['n_train']} test={pair_state['n_test']} "
    f"feats={len(pair_state['selected_features'])}"
)
print(f"  holdout  AUC={m['auc']:.4f} Brier={m['brier']:.4f} ECE={m['ece']:.4f}")
if cv.get("n_folds"):
    print(
        f"  {cv['n_folds']}-fold CV AUC={cv['auc_mean']:.4f}"
        f" (+/-{cv['auc_std']:.4f}) ECE={cv['ece_mean']:.4f}"
    )

run_id = pair_model.save_pair_run(
    conn, pair_state, n_labeled_pairs=int(labels["bus_id"].nunique())
)
conn.commit()
print(f"pair model persisted as run #{run_id}")

/home/victor/repos/opa-database/ml/bus_matching_model/app/pair_features.py:193: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dictionary = pd.read_sql(_DICTIONARY_SQL, conn)
/home/victor/repos/opa-database/ml/bus_matching_model/app/pair_features.py:213: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  running = pd.read_sql(_BUS_RUNNING_DAYS_SQL, conn)
/home/victor/repos/opa-database/ml/bus_matching_model/app/pair_model.py:87: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


120180 candidate pairs, 84 pair labels (77 correct, 7 unsure)
pair model: train=4613 test=2005 feats=73
  holdout  AUC=1.0000 Brier=0.0002 ECE=0.0004
  5-fold CV AUC=1.0000 (+/-0.0000) ECE=0.0003
pair model persisted as run #13


## Build the final table

`THRESHOLD` is the ship threshold picked from `precision_at_threshold` /
`audit_precision` while labeling, not a guess -- see the pair-labeler
UI's "When to stop" panel for the measured precision behind this
number.


In [5]:
THRESHOLD = 0.90

ranked = pair_model.rank_pairs(pairs, pair_state)
final_table = final_output.build_final_pairs(
    conn, day, ranked, labels, excluded, threshold=THRESHOLD
)
print(f"{len(final_table)} rows, {final_table['bus_id'].nunique()} distinct buses")
final_table["method"].value_counts()

1734 rows, 1730 distinct buses


method
pair_model               1345
excluded_no_avl           249
hand_confirmed             77
no_candidates              48
split_detected              8
no_evidence                 4
needs_review                2
resolved_after_review       1
Name: count, dtype: int64

In [6]:
final_output.save_final_pairs(conn, final_table)
conn.commit()
print(f"wrote {len(final_table)} rows to ml.bus_matching_final_pairs")

wrote 1734 rows to ml.bus_matching_final_pairs


## Sanity checks


In [7]:
total_ran = pd.read_sql(
    "SELECT count(DISTINCT bus_id) FROM ml.trip_validity_final WHERE is_valid;", conn
).iloc[0, 0]
distinct_in_table = pd.read_sql(
    "SELECT count(DISTINCT bus_id) FROM ml.bus_matching_final_pairs;", conn
).iloc[0, 0]
print(f"buses that ran: {total_ran}")
print(f"distinct buses in final table: {distinct_in_table}")
print(
    "MATCH (every bus that ran got exactly one bucket)"
    if total_ran == distinct_in_table
    else "MISMATCH -- investigate before trusting this table"
)

# No bus should ever have overlapping date ranges across its rows --
# a real split has disjoint intervals, everything else is a single
# full-month row.
overlap = pd.read_sql(
    """
    SELECT count(*) FROM ml.bus_matching_final_pairs a
    JOIN ml.bus_matching_final_pairs b
      ON a.bus_id = b.bus_id AND a.start_date < b.start_date
    WHERE a.end_date >= b.start_date;
    """,
    conn,
).iloc[0, 0]
print(f"overlapping intervals for the same bus (must be 0): {overlap}")

/tmp/ipykernel_1284677/1301532072.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_ran = pd.read_sql(


buses that ran: 1730
distinct buses in final table: 1730
MATCH (every bus that ran got exactly one bucket)
overlapping intervals for the same bus (must be 0): 0


/tmp/ipykernel_1284677/1301532072.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  distinct_in_table = pd.read_sql(
/tmp/ipykernel_1284677/1301532072.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overlap = pd.read_sql(


In [8]:
print(
    pd.read_sql(
        "SELECT method, count(*) AS n_rows, count(DISTINCT bus_id) AS n_buses "
        "FROM ml.bus_matching_final_pairs GROUP BY method ORDER BY n_rows DESC;",
        conn,
    ).to_string(index=False)
)

               method  n_rows  n_buses
           pair_model    1345     1345
      excluded_no_avl     249      249
       hand_confirmed      77       77
        no_candidates      48       48
       split_detected       8        4
          no_evidence       4        4
         needs_review       2        2
resolved_after_review       1        1


/tmp/ipykernel_1284677/3437356920.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


### Split detection, checked against the four buses found by hand

Each should show two intervals whose boundary matches the day-score
cutover confirmed by hand while labeling: 12225 at Nov 20/21, 12502 at
Nov 13/14 (the one genuine overlap day, resolved to the higher score),
35252 at Nov 21/23 (no service on the 22nd), 35403 at Nov 16/17.


In [9]:
known_swaps = ["12225", "12502", "35252", "35403"]
splits = pd.read_sql(
    "SELECT bus_id, device_id, start_date, end_date, n_days_with_data, notes "
    "FROM ml.bus_matching_final_pairs WHERE method = 'split_detected' "
    "ORDER BY bus_id, start_date;",
    conn,
)
print(splits.to_string(index=False))
detected = set(splits["bus_id"])
print(
    "MATCH (all four hand-caught swaps auto-detected as splits)"
    if detected == set(known_swaps)
    else f"MISMATCH -- expected {known_swaps}, detected {sorted(detected)}"
)

bus_id     device_id start_date   end_date  n_days_with_data                                notes
 12225 ep1-428103715 2023-11-01 2023-11-20                13                                     
 12225 ep1-428108761 2023-11-21 2023-11-30                 9                                     
 12502 ep1-428115555 2023-11-01 2023-11-13                 8                                     
 12502 ep1-428112320 2023-11-14 2023-11-30                13 overlap day resolved to higher score
 35252 ep1-428106939 2023-11-01 2023-11-21                13                                     
 35252 ep1-428103933 2023-11-23 2023-11-30                 6                                     
 35403 ep1-428108461 2023-11-01 2023-11-16                13                                     
 35403 ep1-428107060 2023-11-21 2023-11-30                 8                                     
MATCH (all four hand-caught swaps auto-detected as splits)


/tmp/ipykernel_1284677/4274453616.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  splits = pd.read_sql(


### The remaining unresolved buckets

`needs_review` is the honest residue: an `unsure` label where the
day-score pattern itself does not show a clean single-device or
clean-swap shape, so no automatic answer is defensible -- these need a
human look, same as any bus still below threshold.


In [10]:
for method in ("needs_review", "resolved_after_review", "no_evidence"):
    sub = pd.read_sql(
        "SELECT bus_id, device_id, notes FROM ml.bus_matching_final_pairs "
        "WHERE method = %(m)s ORDER BY bus_id;",
        conn,
        params={"m": method},
    )
    print(f"--- {method} ({len(sub)}) ---")
    print(sub.to_string(index=False) if not sub.empty else "(none)")
    print()

--- needs_review (2) ---
bus_id device_id                               notes
 20290       nan no confident days for any candidate
 21517       nan no confident days for any candidate

--- resolved_after_review (1) ---
bus_id     device_id                                                                          notes
 35322 ep1-428105660 unsure label, but only one device was ever confident -- not actually ambiguous

--- no_evidence (4) ---
bus_id device_id notes
 30162       nan      
 30713       nan      
 30915       nan      
 36975       nan      



/tmp/ipykernel_1284677/3100884984.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sub = pd.read_sql(
/tmp/ipykernel_1284677/3100884984.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sub = pd.read_sql(
/tmp/ipykernel_1284677/3100884984.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sub = pd.read_sql(


In [11]:
below = pd.read_sql(
    "SELECT bus_id, device_id, confidence, n_days_with_data "
    "FROM ml.bus_matching_final_pairs WHERE method = 'below_threshold' "
    "ORDER BY confidence DESC;",
    conn,
)
print(
    f"below_threshold: {len(below)} buses -- real evidence, just not enough of it yet"
)
below

below_threshold: 0 buses -- real evidence, just not enough of it yet


/tmp/ipykernel_1284677/1867270476.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  below = pd.read_sql(


,bus_id,device_id,confidence,n_days_with_data


## The other half: unclaimed devices

`bus_matching_final_pairs` is bus-centric -- for every bus, which
device. That leaves the reverse question unanswered: a device can be
genuinely active all month and still never be claimed, either because
it lost a competition for a bus another device won, or because
blocking never considered it for anyone. `device_coverage.py` answers
that, with the same never-silently-drop discipline as the bus-side
table.


In [12]:
import device_coverage

unclaimed = device_coverage.build_unclaimed_devices(conn, ranked, excluded)
print(f"{len(unclaimed)} active devices claimed by no bus")
unclaimed["reason"].value_counts()

/home/victor/repos/opa-database/ml/bus_matching_model/app/device_coverage.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  active = pd.read_sql(_ACTIVE_DEVICES_SQL, conn)
/home/victor/repos/opa-database/ml/bus_matching_model/app/device_coverage.py:87: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candidates = pd.read_sql(_CANDIDATE_BUSES_SQL, conn)
/home/victor/repos/opa-database/ml/bus_matching_model/app/device_coverage.py:90: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dictionary = pd.read_sql(_DICTIONARY_BUSES_SQL,

62 active devices claimed by no bus


reason
never_blocked       44
lost_competition    18
Name: count, dtype: int64

In [13]:
device_coverage.save_unclaimed_devices(conn, unclaimed)
conn.commit()
print(f"wrote {len(unclaimed)} rows to ml.bus_matching_unclaimed_devices")

wrote 62 rows to ml.bus_matching_unclaimed_devices


A device with real activity and a dictionary hit that still lost is
worth a specific look -- it is exactly the kind of case the residual
pass should reconcile with the bus-side `no_evidence`/`needs_review`
rows, not just more unexplained absence.


In [14]:
MEANINGFUL_ACTIVITY_PINGS = 10_000  # a real, well-tracked device, not a one-off blip

worth_a_look = unclaimed[
    (unclaimed["in_dictionary"]) & (unclaimed["n_pings"] > MEANINGFUL_ACTIVITY_PINGS)
].sort_values("n_pings", ascending=False)
print(f"{len(worth_a_look)} active, dictionary-backed, still-unclaimed devices")
worth_a_look[
    [
        "device_id",
        "n_pings",
        "n_days_active",
        "reason",
        "dictionary_bus_ids",
        "best_candidate_bus_id",
        "best_candidate_score",
    ]
]

5 active, dictionary-backed, still-unclaimed devices


,device_id,n_pings,n_days_active,reason,dictionary_bus_ids,best_candidate_bus_id,best_candidate_score
0,ep1-428109738,113914,30,lost_competition,30162,30162,0.000069
7,ep1-428111212,80398,30,lost_competition,30713,30713,0.000072
20,ep1-428106248,17189,8,lost_competition,35450,35226,0.000027
24,ep1-428112540,14208,6,lost_competition,12206,12317,0.000027
28,ep1-428113356,11005,3,lost_competition,35230,35230,0.000029


## Next

- `needs_review` + `below_threshold` + `no_evidence` buses: candidates
  for the residual-matching pass (exhaustive scoring of unassigned
  buses against unassigned devices), not more queue labeling.
- `no_candidates` (48 buses): blocking found nothing at all -- same
  residual pass, different starting point (no candidate to even score).
- `excluded_no_avl` (249 buses, all 67-prefix): structurally out of
  scope, nothing to do.
- Unclaimed devices with real activity and a dictionary hit (see above) are a second, concrete entry point into the same residual pass -- not a separate problem.


In [15]:
conn.close()